# Simple: Date Field Analysis with PAMOLA.CORE

**Goal**: Learn date field profiling basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze date fields for validation and quality using execute()
- Compare before/after results
- Understand date distributions and anomalies

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 01_date_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.date import DateOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records including date fields suitable for profiling

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with birth dates
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'birth_date': [
            '1985-03-15', '1990-07-22', '1978-11-30', '1995-01-10', '1982-05-18',
            '1988-09-25', '1992-12-05', '1975-04-14', '1998-08-20', '1980-02-28',
            '1987-06-17', '1993-10-08', '1976-03-22', '1991-11-15', '1984-07-09',
            '1989-01-30', '1994-05-12', '1979-09-03', '1996-12-25', '1983-04-07'
        ],
        'registration_date': [
            '2020-01-15', '2020-02-20', '2020-03-10', '2020-04-05', '2020-05-12',
            '2020-06-18', '2020-07-22', '2020-08-14', '2020-09-09', '2020-10-25',
            '2021-01-11', '2021-02-16', '2021-03-20', '2021-04-28', '2021-05-30',
            '2021-06-15', '2021-07-08', '2021-08-19', '2021-09-22', '2021-10-30'
        ],
        'name': [
            'Alice', 'Bob', 'Charlie', 'Diana', 'Eve',
            'Frank', 'Grace', 'Henry', 'Iris', 'Jack',
            'Karen', 'Leo', 'Mia', 'Nathan', 'Olivia',
            'Paul', 'Quinn', 'Rachel', 'Sam', 'Tina'
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field name** for processing

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "snapshot_date"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "snapshot_date"  # Field to analyze - CUSTOMIZE THIS!
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].unique()[:5])}")
print(f"  Data type: {df[field_name].dtype}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="date_profiling",
    description=f"Profiling of '{field_name}' date field",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create DateOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_name=field_name`: Field to analyze
- `min_year=1940`: Minimum valid year for anomaly detection
- `max_year=2005`: Maximum valid year for anomaly detection
- `is_birth_date=True`: Calculate age distributions
- `id_column=None`: Optional group analysis column
- `uid_column=None`: Optional UID consistency check column
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = DateOperation(
    # Core parameters
    field_name=field_name,
    
    # Date validation parameters
    min_year=2000,                   # Minimum valid year
    max_year=2024,                   # Maximum valid year
    
    # Analysis parameters
    is_birth_date=True,              # Calculate age distributions
    id_column=None,                  # Optional: group analysis
    uid_column=None,                 # Optional: UID consistency
    profile_type='date',             # Profile type for organization
    
    # Processing settings
    chunk_size=10000,
    use_dask=False,
    use_vectorization=False,
    parallel_processes=1,
    npartitions=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,     # Create visualization artifacts
    save_output=True,                # Save results to files
    force_recalculation=False        # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Min year:         {operation.min_year}")
print(f"  Max year:         {operation.max_year}")
print(f"  Is birth date:    {operation.is_birth_date}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing date field analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the analysis output from task_dir
- Review metrics and statistics
- Examine date distributions and anomalies

**Generated artifacts:**
- Statistics JSON in output/
- Metrics JSON in metrics/
- Visualizations in visualizations/
- Anomalies CSV in dictionaries/ (if any)

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load statistics file
output_files = list(task_dir.glob('output/*_stats_*.json'))
if output_files:
    stats_file = output_files[0]
    
    with open(stats_file, 'r') as f:
        stats = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 DATE FIELD ANALYSIS RESULTS")
    print("=" * 80)
    
    # Basic statistics
    print("\n📈 Data Quality Metrics:")
    print(f"  Total records:    {stats.get('total_records', 0):,}")
    print(f"  Null count:       {stats.get('null_count', 0):,}")
    print(f"  Valid dates:      {stats.get('valid_count', 0):,}")
    print(f"  Invalid dates:    {stats.get('invalid_count', 0):,}")
    print(f"  Fill rate:        {stats.get('fill_rate', 0):.2f}%")
    print(f"  Valid rate:       {stats.get('valid_rate', 0):.2f}%")
    
    # Date range
    if 'min_date' in stats and 'max_date' in stats:
        print("\n📅 Date Range:")
        print(f"  Earliest date:    {stats['min_date']}")
        print(f"  Latest date:      {stats['max_date']}")
    
    # Anomalies
    if 'anomalies' in stats:
        print("\n⚠️  Anomalies Detected:")
        anomalies = stats['anomalies']
        total_anomalies = sum(anomalies.values())
        if total_anomalies > 0:
            for anomaly_type, count in anomalies.items():
                if count > 0:
                    print(f"  {anomaly_type.replace('_', ' ').title():20s}: {count}")
            print(f"  {'Total':20s}: {total_anomalies}")
        else:
            print("  ✓ No anomalies found")
    
    # Age statistics (if birth date)
    if 'age_statistics' in stats:
        print("\n👤 Age Statistics:")
        age_stats = stats['age_statistics']
        print(f"  Min age:          {age_stats.get('min_age', 'N/A')}")
        print(f"  Max age:          {age_stats.get('max_age', 'N/A')}")
        print(f"  Mean age:         {age_stats.get('mean_age', 'N/A'):.1f}")
        print(f"  Median age:       {age_stats.get('median_age', 'N/A'):.1f}")
    
    # Year distribution
    if 'year_distribution' in stats:
        print("\n📊 Year Distribution (Top 10):")
        year_dist = stats['year_distribution']
        sorted_years = sorted(year_dist.items(), key=lambda x: int(x[0]))
        for year, count in sorted_years[:10]:
            bar = '█' * min(50, count * 5)
            print(f"  {year}: {bar} {count}")
    
    # Display result metrics
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    if result.metrics:
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Date field '{field_name}' successfully analyzed!")
else:
    print("⚠️  No statistics file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Statistics JSON
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
├── dictionaries/     # Anomalies CSV (if any)
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'dictionaries', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and effectiveness metrics
2. **Statistics Data**: Date field analysis with distributions and anomalies
3. **Visualizations**: Charts showing year, month, day-of-week, and age distributions
4. **Anomalies CSV**: Detailed list of detected anomalies (if any)

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display key metrics
            print("\n📊 KEY METRICS:")
            metric_keys = ['total_rows', 'null_count', 'valid_count', 'fill_rate', 
                          'valid_rate', 'min_date', 'max_date', 'min_age', 'max_age', 
                          'mean_age', 'median_age', 'anomalies_count']
            
            for key in metric_keys:
                if key in metrics:
                    value = metrics[key]
                    if isinstance(value, float):
                        print(f"   {key:20s}: {value:.2f}")
                    else:
                        print(f"   {key:20s}: {value}")
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. STATISTICS DATA (NEWEST)
print("\n\n2️⃣ STATISTICS DATA:")
print("=" * 80)

output_dir = task_dir / "output"

if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    stats_files = sorted(
        output_dir.glob("*_stats_*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not stats_files:
        print("⚠️  No statistics files found")
    else:
        latest_stats_file = stats_files[0]
        mtime = datetime.fromtimestamp(latest_stats_file.stat().st_mtime)

        print(f"📄 File     : {latest_stats_file.name}")
        print(f"🕒 Modified : {mtime:%Y-%m-%d %H:%M:%S}")
        print(f"📦 Size     : {latest_stats_file.stat().st_size / 1024:.1f} KB\n")

        try:
            with open(latest_stats_file, "r", encoding="utf-8") as f:
                stats = json.load(f)

            # BASIC COUNTS
            print("📌 BASIC STATISTICS")
            print("-" * 80)
            print(f"Total records : {stats.get('total_records')}")
            print(f"Fill rate     : {stats.get('fill_rate')}%")
            print(f"Valid rate    : {stats.get('valid_rate')}%")
            print(f"Date range    : {stats.get('min_date')} → {stats.get('max_date')}")
            print()

            # YEAR DISTRIBUTION
            if "year_distribution" in stats:
                print("📊 Year Distribution")
                print("-" * 80)
                for year, count in sorted(
                    stats["year_distribution"].items(),
                    key=lambda x: int(x[0]),
                ):
                    bar = "█" * min(50, count)
                    print(f"{year}: {bar} {count}")
                print()

            # MONTH DISTRIBUTION
            if "month_distribution" in stats:
                print("📊 Month Distribution")
                print("-" * 80)
                month_names = [
                    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
                ]
                for month, count in sorted(
                    stats["month_distribution"].items(),
                    key=lambda x: int(x[0]),
                ):
                    name = month_names[int(month) - 1]
                    bar = "█" * min(50, count)
                    print(f"{name}: {bar} {count}")
                print()

            # DAY OF WEEK DISTRIBUTION
            if "day_of_week_distribution" in stats:
                print("📊 Day of Week Distribution")
                print("-" * 80)
                for day, count in stats["day_of_week_distribution"].items():
                    bar = "█" * min(50, count)
                    print(f"{day:<10}: {bar} {count}")
                print()

            # AGE ANALYSIS (SEMANTIC VALIDATION)
            if "age_statistics" in stats:
                print("👤 Age Analysis (Semantic Validation)")
                print("-" * 80)

                age_stats = stats["age_statistics"]
                anomalies = stats.get("anomalies", {})
                total = stats.get("total_records", 1)

                print(f"Min age   : {age_stats.get('min_age')}")
                print(f"Max age   : {age_stats.get('max_age')}")
                print(f"Mean age  : {age_stats.get('mean_age'):.2f}")
                print(f"Median    : {age_stats.get('median_age')}")

                too_young = anomalies.get("too_young", 0)
                if too_young > 0:
                    ratio = too_young / total
                    print(f"\n🚨 TOO_YOUNG anomaly: {too_young} records ({ratio:.1%})")

                    if ratio > 0.2:
                        print("⚠️  High anomaly ratio detected")
                        print("   Possible semantic mismatch:")
                        print("   - snapshot_date misused as birth_date")
                        print("   - age derivation is NOT VALID")
                        print("   👉 Age-based analytics SHOULD BE DISABLED")

                    examples = stats.get("too_young_examples", [])[:5]
                    if examples:
                        print("\nExamples:")
                        for ex in examples:
                            print(f"  - {ex}")

                # Only show age distribution if logically valid
                if too_young == 0 and "age_distribution" in stats:
                    print("\n📊 Age Distribution")
                    print("-" * 80)
                    for group, count in sorted(
                        stats["age_distribution"].items(),
                        key=lambda x: int(x[0].split("-")[0]),
                    ):
                        bar = "█" * min(50, count)
                        print(f"{group}: {bar} {count}")

            # FINAL QUALITY FLAG
            semantic_valid = stats.get("anomalies", {}).get("too_young", 0) == 0
            print("\n✅ SEMANTIC VALIDATION RESULT")
            print("-" * 80)
            print(f"Age semantics valid: {semantic_valid}")

        except Exception as e:
            print(f"❌ Error reading statistics file: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                viz_name = viz_file.stem.replace('_', ' ').replace(field_name, '').strip().title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

# 4. ANOMALIES CSV (if exists)
print("\n\n4️⃣ ANOMALIES DATA:")
print("-" * 80)

dict_dir = task_dir / 'dictionaries'
if dict_dir.exists():
    anomaly_files = list(dict_dir.glob('*anomalies*.csv'))
    
    if not anomaly_files:
        print("   ✓ No anomalies detected")
    else:
        # Sort by modification time (newest first)
        anomaly_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_anomaly_file = anomaly_files[0]
        
        mtime = datetime.fromtimestamp(latest_anomaly_file.stat().st_mtime)
        print(f"✓ Found {len(anomaly_files)} anomaly file(s)")
        print(f"📄 Loading LATEST: {latest_anomaly_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        try:
            anomalies_df = pd.read_csv(latest_anomaly_file)
            print(f"📊 Total anomalies: {len(anomalies_df)}")
            print(f"\nAnomaly types breakdown:")
            print(anomalies_df['anomaly_type'].value_counts())
            print(f"\nFirst 10 anomalies:")
            print(anomalies_df.head(10).to_string())
        except Exception as e:
            print(f"❌ Error reading anomalies: {e}")
else:
    print("   ℹ️  Dictionaries directory not found")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure date profiling with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Date field analysis validates format and detects anomalies
- Distributions show patterns across years, months, and days
- Age calculations available for birth date fields
- Anomalies include: too old, too young, future dates, invalid formats
- Visualizations help identify data quality issues

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_date_analyzer_advanced.ipynb`** includes:
- Group analysis (detect changes within ID groups)
- UID consistency checks (find inconsistencies by person)
- Custom date format handling
- Parallel processing for large datasets
- Dask integration for distributed computing

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)